# Spikecurate

_via regression model_

**author**: laquitainesteeve@gmail.com

**purpose**: curate the "good" units (>80% agreement score) amongst the sorted single-units, for the biophysical simulation in the spontaneous regime, with the two best models so far: a logistic regression (interpretable) and an extra-trees classifier (best overall). Both use the 10 quality metrics plus new features of the unit firing pattern, amplitude distribution, footprint and similarity with other units.

**tested on**:

* Ubuntu 24.04.1 LTS (32 cores, 188 GB RAM, Intel(R) Core(TM) i9-14900K ＠3.2 GHz/5.8 GHz) with RTX 5090 GPU with 40GB VRAM (GPU not needed)

**Method**
- kilosort4's sorting extractor was generated with rtx5090 GPU, so the results slightly change from the original Tesla V100
- a dataset dataframe that contains the sorted single-units (indices), their quality metrics and their quality label ("good" or "bad" units evaluated with our ground truth, columns)

**Execution time**: 3 min


## Setup 

1. Create and activate envs/spikebias.yml

In [1]:
# automatically reload notebook
%load_ext autoreload
%autoreload 2

# import built-in packages
import os

# limit BLAS/OpenMP threads per worker, BEFORE numpy is imported (restart the kernel
# if numpy was already imported): the PCA runs ~20x faster than with default threads
# Each worker runs many small PCA fits, each fit makes many BLAS/LAPACK calls, by 
# default each call can use up to 32 threads. We capped that to 1 thread, which 
# substantially removed overhead and sped up compute.
# 2 min instead of 1.5h
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[var] = "1"
    
import shutil
import pickle
import copy
import random
import pandas as pd
import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.core.template_tools as ttools
from spikeinterface import comparison
from spikeinterface.curation import remove_excess_spikes
from spikeinterface import extract_waveforms
from spikeinterface.postprocessing import compute_principal_components
from spikeinterface.qualitymetrics import compute_quality_metrics as qm
import statsmodels.api as sm
from statsmodels.tools.validation import float_like
import torch
import numpy as np
from matplotlib import pyplot as plt
import sklearn
import seaborn as sns
from sklearn import metrics
from sklearn.preprocessing import StandardScaler
import warnings
import spikecurate 

# silence irrelevant warnings
warnings.filterwarnings("ignore", message="There is no Probe attached to this recording.*")
warnings.filterwarnings("ignore", message="Versions are not the same.*")

# spikeinterface version
print("Spikeinterface", si.__version__)

# set project path
proj_path = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(proj_path)
os.environ["PYTHONPATH"] = proj_path     # worker processes (joblib) must be able to import the project

# import custom package
# (helpers that read sorting_quality.csv are not in the spikecurate package)
from src.nodes.models.Flc import dataloader
from src.nodes.models.Flc import extra_features as ef             # candidate features
from src.nodes.models.Flc import curation_models as cm           # curation models + nested cross-validation

# experiment parameters
EXP = "NS"
SORTER = "KS4"

# waveform extraction parameters
FREQ_MIN = 300
REFERENCE = "global"
OPERATOR = "median"
MS_BEFORE = 6
MS_AFTER = 6
LOAD_WE = False
DURATION_SEC = 600          # 10 minutes recording
MAX_SPIKES_PER_UNIT = 500   # max number of spikes used per unit
SEED = 0
N_COMP_PCA = 5              # number of pca components
SPARSE = True
OVERWRITE = True
WE_JOB_KWARGS = dict(n_jobs=20, chunk_duration="1s", progress_bar=True)
job_kwargs = dict(n_jobs=-1, progress_bar=True)

CHUNKS = 80000 # for waveform extractor

# model parameters
model_prms = {"seeds": np.arange(0,100,1), "scale_data": False}

# recording extractor
RECORDING_PATH = "dataset/00_raw/recording_npx_spont/"

# kilosort sorting extractor
SORTING_KS4_PATH = "dataset/01_intermediate/sorting/npx_spont/SortingKS4_10m/"

# ground truth extractors
SORTING_TRUE = "dataset/00_raw/ground_truth_npx_spont"
SORTING_TRUE_10mn = "dataset/00_raw/ground_truth_npx_spont_10m"

# waveform extractors
# WE_PATH = "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m/"
WE_SINGLE_U_PATH = "dataset/01_intermediate/waveforms/npx_spont/SortingKS4_10m_single_units"

# unit classification
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality.csv"

# figure parameters
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5 # typically between 0.5 and 1
plt.rcParams["axes.linewidth"] = 0.5 
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1

# figure legend parameters
legend_cfg = {"frameon": False, "handletextpad": 0.5}
tight_layout_cfg = {"pad": 0.001}
LG_FRAMEON = False              # no legend frame

Spikeinterface 0.100.8


## Load datasets

* Execution time: 27 ms

### Recording extractor

* Execution time: 

In [2]:
%%time

# load and preprocess recording
Recording = si.load_extractor(RECORDING_PATH)
Recording.set_property('location', Recording.get_property('location')[:,:2])
Recording = spre.highpass_filter(Recording, freq_min=FREQ_MIN)
Recording = spre.common_reference(Recording, reference=REFERENCE, operator=OPERATOR)
# keep the first 10 min so that the recording duration matches the sorting
# (duration-dependent quality metrics: firing_rate, firing_range, presence_ratio)
Recording = Recording.frame_slice(start_frame=0, end_frame=int(Recording.get_sampling_frequency() * DURATION_SEC))
display(Recording)

FrameSliceRecording: 384 channels - 40.0kHz - 1 segments - 24,000,000 samples 
                     600.00s (10.00 minutes) - float32 dtype - 34.33 GiB

CPU times: user 3.96 ms, sys: 0 ns, total: 3.96 ms
Wall time: 3.94 ms


### Ground truth extractor

* Execution time: 

In [3]:
%%time

# load ground truth
SortingTrue = si.load_extractor(SORTING_TRUE)
SortingTrue = SortingTrue.frame_slice(start_frame=0, end_frame=int(Recording.get_sampling_frequency() * DURATION_SEC))
SortingTrue.save(folder=SORTING_TRUE_10mn, overwrite=True)
display(SortingTrue)

FrameSliceSorting: 1388 units - 1 segments - 40.0kHz

CPU times: user 234 ms, sys: 7.84 ms, total: 242 ms
Wall time: 242 ms


### Sorting extractor

* Execution time: 20 ms

In [4]:
%%time

# load sorting
Sorting = si.load_extractor(SORTING_KS4_PATH)
Sorting = remove_excess_spikes(Sorting, Recording)
Sorting = Sorting.frame_slice(start_frame=0, end_frame=Recording.get_sampling_frequency() * DURATION_SEC)
display(Sorting)

FrameSliceSorting: 520 units - 1 segments - 40.0kHz

CPU times: user 1.3 ms, sys: 41 µs, total: 1.34 ms
Wall time: 1.32 ms


### Waveform extractor (Sorted units)

* Execution time: 2.5 min

In [5]:
%%time 

# select sorted single-units amongst single- and multi-units (KSLABEL="good" for kilosort4)
single_unit_ids = Sorting.unit_ids[np.where(Sorting.get_property("KSLabel") == "good")[0]]
Sorting_single_units = Sorting.select_units(unit_ids=single_unit_ids)

# save their waveforms
We_single_u = extract_waveforms(Recording, Sorting_single_units, WE_SINGLE_U_PATH, mode="folder", sparse=SPARSE, ms_before=MS_BEFORE,
                    ms_after=MS_AFTER, max_spikes_per_unit=MAX_SPIKES_PER_UNIT, unit_batch_size=CHUNKS,
                    overwrite=OVERWRITE, seed=SEED, **WE_JOB_KWARGS)


# save waveform PCA data (the extractor is updated)
pca = compute_principal_components(waveform_extractor=We_single_u, n_components=N_COMP_PCA, mode="by_channel_local", **WE_JOB_KWARGS)

extract waveforms shared_memory multi buffer:   0%|          | 0/600 [00:00<?, ?it/s]

extract waveforms memmap multi buffer:   0%|          | 0/600 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/184 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/184 [00:00<?, ?it/s]

CPU times: user 24.5 s, sys: 53 s, total: 1min 17s
Wall time: 2min 34s


### Single-unit ids and labels (good/gad)

In [6]:
%%time

# label "good" and "bad" sorted single-units to train the classifier
good_unit_ids, bad_unit_ids = dataloader.get_good_and_bad_units(QUALITY_PATH, exp="NS", sorter="KS4")

# report
print("\nSingle-unit ids:", single_unit_ids)
print("\nGood unit ids:", good_unit_ids)
print("\nBad unit ids:", bad_unit_ids)


Single-unit ids: ['26' '29' '40' '41' '42' '45' '48' '53' '54' '55' '56' '58' '60' '61'
 '66' '71' '76' '77' '81' '83' '84' '85' '86' '90' '96' '102' '104' '105'
 '108' '109' '110' '113' '116' '117' '120' '121' '122' '126' '127' '128'
 '130' '135' '140' '142' '143' '146' '149' '152' '157' '158' '160' '162'
 '165' '166' '172' '183' '189' '193' '197' '198' '199' '201' '202' '207'
 '211' '213' '214' '221' '223' '227' '231' '239' '240' '241' '247' '249'
 '250' '252' '254' '262' '263' '264' '269' '270' '277' '280' '281' '283'
 '284' '286' '289' '291' '292' '296' '297' '299' '300' '302' '304' '307'
 '308' '310' '315' '316' '318' '319' '321' '323' '325' '328' '333' '339'
 '341' '342' '345' '346' '348' '350' '358' '365' '368' '371' '374' '380'
 '381' '384' '385' '386' '389' '390' '391' '393' '396' '398' '399' '400'
 '401' '402' '405' '408' '410' '411' '413' '420' '424' '426' '429' '432'
 '435' '438' '440' '442' '445' '446' '448' '449' '450' '451' '452' '453'
 '455' '458' '459' '460' '461' '46

### Unit-tests

In [7]:
%%time

# sanity check units are correct
single_unit_ids_from_file = dataloader.get_sorted_single_unit_ids_from_file(QUALITY_PATH, EXP, SORTER)
assert len(np.intersect1d(single_unit_ids_from_file, single_unit_ids)) == len(single_unit_ids_from_file), "Units are missing or differ"

CPU times: user 4.79 ms, sys: 0 ns, total: 4.79 ms
Wall time: 4.79 ms


## Engineer unit quality metrics

* Execution time: 2 min

In [8]:
%%time

# engineer a dataset of quality metrics (cols) per cell (rows)
data = spikecurate.load_dataset(single_unit_ids, good_unit_ids, bad_unit_ids, Sorting_single_units, SortingTrue, We_single_u, load_qm_if_exists=False, delta_time=1.3, job_kwargs=WE_JOB_KWARGS)

extract amplitudes:   0%|          | 0/600 [00:00<?, ?it/s]

  0%|          | 0/184 [00:00<?, ?it/s]

extract amplitudes:   0%|          | 0/600 [00:00<?, ?it/s]

CPU times: user 13.5 s, sys: 6.36 s, total: 19.8 s
Wall time: 1min 23s


In [9]:
%%time 

# measure correlations between features (less is better)
data["dataset"].corr()

CPU times: user 62 µs, sys: 92 µs, total: 154 µs
Wall time: 155 µs


,amplitude_cutoff,firing_range,firing_rate,isi_violations_ratio,rp_contamination,rp_violations,sd_ratio,snr,silhouette,mad_ratio,quality_label
amplitude_cutoff,1.000000,-0.098276,-0.155109,0.294182,0.061312,-0.032527,0.460140,0.599839,0.382980,0.255123,-0.101853
firing_range,-0.098276,1.000000,0.577485,-0.047203,0.129424,0.423390,0.055066,-0.256396,-0.325428,0.224384,-0.378831
firing_rate,-0.155109,0.577485,1.000000,-0.080729,0.046400,0.429786,-0.113885,-0.208821,-0.239668,-0.004278,0.066000
isi_violations_ratio,0.294182,-0.047203,-0.080729,1.000000,0.558257,0.182912,0.225188,0.287679,0.176475,0.067380,0.023906
rp_contamination,0.061312,0.129424,0.046400,0.558257,1.000000,0.500537,0.105478,-0.002048,-0.030830,0.064045,-0.019728
rp_violations,-0.032527,0.423390,0.429786,0.182912,0.500537,1.000000,0.005510,-0.078501,-0.099176,0.015567,-0.043278
sd_ratio,0.460140,0.055066,-0.113885,0.225188,0.105478,0.005510,1.000000,0.444656,0.090362,0.875050,-0.283486
snr,0.599839,-0.256396,-0.208821,0.287679,-0.002048,-0.078501,0.444656,1.000000,0.561631,0.105750,0.217487
silhouette,0.382980,-0.325428,-0.239668,0.176475,-0.030830,-0.099176,0.090362,0.561631,1.000000,-0.126620,0.153130
mad_ratio,0.255123,0.224384,-0.004278,0.067380,0.064045,0.015567,0.875050,0.105750,-0.126620,1.000000,-0.438458


## Model: train/detect >80% accurate units

Two models, evaluated with nested cross-validation (10 x repeated stratified 4-fold, 75% train / 25% test). The decision threshold of each model is chosen on the training units only: the lowest threshold whose out-of-fold precision reaches 0.92 (the earlier model's precision).

* **logistic regression**: 10 quality metrics + firing pattern (ISI CV and local variation, Fano factor, empty fraction and trend of 5 s bins, short ISI fraction)
* **extra trees** (best overall): 10 quality metrics + all new features (amplitude distribution, firing pattern, spatial footprint and depth, waveform consistency, similarity with other units, Kilosort cluster properties; see src/nodes/models/Flc/extra_features.py)

In [10]:
%%time

# new features of each sorted single-unit (no ground truth is used)
extra = ef.compute_extra_features(We_single_u, duration=DURATION_SEC)

# dataset: 10 quality metrics + new features (units with a missing feature are dropped; silhouette is NaN for a few units)
dataset = data["dataset"].join(extra).dropna()
print("units:", len(dataset), "of", len(data["dataset"]), "| good:", int(dataset["quality_label"].sum()))

units: 182 of 184 | good: 84
CPU times: user 2.5 s, sys: 184 ms, total: 2.68 s
Wall time: 2.68 s


In [11]:
%%time

# the worker processes must be able to import the project
os.environ["PYTHONPATH"] = os.getcwd()

# train & evaluate the two models w/ nested cross-validation
results = cm.nested_cv(dataset, data["predictors"])
display(cm.summarize(results))

BrokenProcessPool: A task has failed to un-serialize. Please ensure that the arguments of the function are all picklable.

## Source data

* precision and recall of each model on the test units of each cross-validation fold (fig5e)
* saved as .xlsx with the .svg figure in figures/8_source_data/fig5e/

In [12]:
# source data: precision and recall of each model in each fold
df = results[["fold", "model", "precision", "recall"]].copy()
display(df.groupby("model", sort=False).agg(["median", "count"]))

NameError: name 'results' is not defined

In [13]:
def plot_source_data(df: pd.DataFrame):
    """Precision and recall of each model across folds (black dots); median and 95% CI in orange"""

    models = list(dict.fromkeys(df["model"]))
    fig, ax = plt.subplots(1, len(models), figsize=(1.1 * len(models) + 0.2, 1.1), sharey=True)
    for a, model in zip(ax, models):
        d = df[df["model"] == model]
        sns.stripplot(ax=a, data=d[["precision", "recall"]], jitter=0.04, color="k", size=3)
        for x, col in enumerate(["precision", "recall"]):
            a.errorbar(x=x, y=np.nanmedian(d[col]), yerr=1.96 * np.std(d[col]) / np.sqrt(len(d[col])), marker="o",
                       color="orange", markeredgecolor="w", markersize=5, zorder=np.inf)
        a.set_title(model, fontsize=6)

        # disconnect axes (R style)
        a.spines[["right", "top"]].set_visible(False)
        a.spines["bottom"].set_position(("axes", -0.05))
        a.yaxis.set_ticks_position("left")
        a.spines["left"].set_position(("axes", -0.05))
        a.set_ylim([0, 1])
    for a in ax[1:]:
        a.spines["left"].set_visible(False); a.set_yticks([])
    plt.subplots_adjust(wspace=0.2)
    return fig, ax

In [14]:
%%time

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

# plot
fig, ax = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig5e", exist_ok=True)
plt.savefig("figures/8_source_data/fig5e/fig5e.svg", bbox_inches="tight", **savefig_cfg)

# save source data
df.to_excel("figures/8_source_data/fig5e/fig5e.xlsx", index=False)

NameError: name 'df' is not defined

In [15]:
%%time

# check: replot from the saved source data
df = pd.read_excel("figures/8_source_data/fig5e/fig5e.xlsx")
fig, ax = plot_source_data(df)

KeyError: 'model'